# ForceLorentzBalance3D — 3D coupled low-Rm MHD `force_lorentz`

`MagneticCoupled3D` · 3D twin of `MagneticCoupled/ForceLorentzBalance`.


## Introduction

3D extrusion of the Lorentz hydrostatic balance: `b=(y,0,0)` is an exact equilibrium (`eta=0`, no transport), `J=(0,0,-1)`, `F=(0,-y,0)` is a pure gradient, so `u=0` and `p=-y^2/2+C`. The case is z-independent; velocity must stay ~0 and the pressure drop `p(0.75)-p(0.45)` must match `-0.18`.


In [ ]:
from trustutils import run
run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read TRUST probe (`.son`) file


In [ ]:
import numpy as np
import os
from trustutils.files import SonFile
BUILD = run.BUILD_DIRECTORY
def read_son(case, name, sub=None):
    if sub is not None:
        cands = [f"{BUILD}/{sub}/{case}_{name}.son",
                 f"{BUILD}/{sub}/PAR_{case}_{name}.son"]
    else:
        cands = [f"{BUILD}/coarse/{case}_{name}.son",
                 f"{BUILD}/{case}_{name}.son"]
    path = next((pp for pp in cands if os.path.exists(pp)), cands[-1])
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)


## Run

Runs on 2 processors (`nbProcs=2` + `case.partition()`).


In [ ]:
NB_PROCS = 2
run.reset()
meshes = {
    "coarse": {"nx": 11, "ny": 11, "nz": 11},
}
cases = {}
for lvl, dic in meshes.items():
    cases[lvl] = run.addCaseFromTemplate(
        "force_lorentz_balance_3d.data", lvl, dic=dic,
        nbProcs=NB_PROCS, excluNR=True)
    cases[lvl].partition()
run.printCases()
run.runCases()
case = cases["coarse"]


## Checks

### #1 — Lorentz hydrostatic balance (pressure difference)


In [ ]:
t1, p1 = read_son(case.dataFileName_, "SONDE_P1")
t2, p2 = read_son(case.dataFileName_, "SONDE_P2")
assert np.allclose(t1, t2), "probe time grids differ"
dp = p2 - p1
expected = -(0.75**2 - 0.45**2) / 2.0
print(f"p2 - p1 = {dp[-1]:.6g}   (analytic {expected:.4g})")
err = abs(dp[-1] - expected) / abs(expected)
print(f"relative error = {err:.3e}")
assert err < 0.05, "pressure balance error too large"
print("OK: pressure balance.")


In [ ]:
# velocity must stay ~0
t, u = read_son(case.dataFileName_, "SONDE_U")
print("max |u| =", np.max(np.abs(u)))
assert np.max(np.abs(u)) < 1e-2, "flow not suppressed"
print("OK: no flow.")


## Computer Performance


In [ ]:
run.tablePerf()